<a href="https://colab.research.google.com/github/Nguyen-The-Thanh/Project-1/blob/main/lm_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
from huggingface_hub import notebook_login

userdata.get("HF_TOKEN")
notebook_login()

In [ ]:
!pip install -U lm-eval
!pip install accelerate sentencepiece datasets evaluate


In [ ]:
import torch
import numpy as np
import random

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

In [ ]:
from transformers import AutoTokenizer

base_model_id = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token


In [ ]:
base_lm = HFLM(
    pretrained=base_model_id,
    tokenizer=tokenizer,
    device="cuda",
    dtype=torch.float16,
    max_length=2048,
)

results_base = evaluator.simple_evaluate(
    model=base_lm,
    tasks=["arc_easy", "hellaswag", "boolq"],
    num_fewshot=0,
    batch_size=2,
)

del base_lm
torch.cuda.empty_cache()


In [ ]:
base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora = PeftModel.from_pretrained(
    base,
    "NguyenTheThanh/llama-3.2-1b-finetuned"
)

lora = lora.merge_and_unload().eval()

lora_lm = HFLM(
    pretrained=lora,
    tokenizer=tokenizer,
    device="cuda",
    dtype=torch.float16,
    max_length=2048,
)

results_lora = evaluator.simple_evaluate(
    model=lora_lm,
    tasks=["arc_easy", "hellaswag", "boolq"],
    num_fewshot=0,
    batch_size=2,

)

del lora_lm, lora, base
torch.cuda.empty_cache()


In [ ]:
full_ft_lm = HFLM(
    pretrained="NguyenTheThanh/llama-3.2-1b-full-finetuned",
    tokenizer=tokenizer,
    device="cuda",
    dtype=torch.float16,
    max_length=2048,
)

results_full = evaluator.simple_evaluate(
    model=full_ft_lm,
    tasks=["arc_easy", "hellaswag", "boolq"],
    num_fewshot=0,
    batch_size=2,
)

del full_ft_lm
torch.cuda.empty_cache()


In [ ]:
print(results_base["results"].keys())
print(results_base["results"]["arc_easy"])

print(results_lora["results"].keys())
print(results_lora["results"]["arc_easy"])

print(results_full["results"].keys())
print(results_full["results"]["arc_easy"])


In [ ]:
def summarize(results):
    return {
        "arc_easy": results["results"]["arc_easy"]["acc_norm,none"],
        "hellaswag": results["results"]["hellaswag"]["acc_norm,none"],
        "boolq": results["results"]["boolq"]["acc,none"],
    }

print("BASE:", summarize(results_base))
print("LoRA:", summarize(results_lora))
print("FULL:", summarize(results_full))
